# LatentMAS-interp — GPU Test Suite

Pulls the repo, installs deps, runs logic tests + end-to-end smoke on real GPU.

In [ ]:
# 1. Clone / update repo
import os
REPO = "https://github.com/spraldev/LatentMAS-interp"
REPO_DIR = "/workspace/LatentMAS-interp"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 2. Install deps
!pip install -q torch transformers accelerate datasets tqdm numpy

In [ ]:
# 3. Confirm GPU is visible
import torch
assert torch.cuda.is_available(), "No GPU found — wrong runtime?"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Logic unit tests (no model, pure math/IO — should be <30s)
!python test_pipeline.py -v

In [ ]:
# 5. W_a gate — run 1 example on Qwen3-4B, check W_a is non-trivial before committing to full run
# If W_a is still identity on 4B, the entire W_a story is dead — stop here.
import os, json
from pathlib import Path
import torch

WA_GATE_DIR = "/workspace/runs/wa_gate"
os.makedirs(WA_GATE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {WA_GATE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --max_samples 1 \
    --latent_space_realign

wa_path = Path(WA_GATE_DIR) / "wa_matrix.pt"
assert wa_path.exists(), f"wa_matrix.pt not found at {wa_path} — run failed"

W = torch.load(wa_path, weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(W)
cond = (S.max() / S.min()).item()
frob = (W - torch.eye(W.shape[0])).norm().item()
frac_near_1 = ((S - 1).abs() < 0.05).float().mean().item()

print(f"W_a shape:           {tuple(W.shape)}")
print(f"cond number:         {cond:.4f}  (want > 1.5)")
print(f"Frob dist from I:    {frob:.4f}  (want > 1.0)")
print(f"frac singvals ≈ 1:   {frac_near_1:.3f}  (want < 0.5)")
print()

WA_OK = cond > 1.5 and frob > 1.0
if WA_OK:
    print("✅ W_a is non-trivial on Qwen3-4B — safe to proceed with full run")
else:
    print("🛑 W_a is still identity on Qwen3-4B")
    print("   Exp A/L/M/Q are all invalid. Do NOT start the full run.")
    print("   Next step: check if --latent_space_realign is wiring correctly in final_run.py")
    raise SystemExit("W_a gate failed — fix W_a before full run")

In [ ]:
# 6. End-to-end smoke on GPU — Qwen3-4B, 5 ex, 2 conditions (~10 min)
# Confirms full pipeline works at real model scale before 10+ hr run.
SMOKE_DIR = "/workspace/runs/smoke"
os.makedirs(SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

In [ ]:
# 7. Check smoke outputs
smoke = Path(SMOKE_DIR)

wa = torch.load(smoke / "wa_matrix.pt", weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(wa)
frob = (wa - torch.eye(wa.shape[0])).norm().item()
print(f"W_a shape: {tuple(wa.shape)}")
print(f"cond number: {S.max()/S.min():.4f}")
print(f"Frob dist from identity: {frob:.4f}")
print()

for cond in ["latent_mas", "single_agent_latent_greedy"]:
    metas = list((smoke / cond / "gsm8k").glob("example_*/metadata.json"))
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"{cond}: {correct}/{len(metas)} correct")

bucket_file = smoke / "buckets" / "gsm8k.json"
if bucket_file.exists():
    from collections import Counter
    rows = json.loads(bucket_file.read_text())
    c = Counter(r["bucket"] for r in rows)
    print(f"\nBuckets: B1={c[1]} B2={c[2]} B3={c[3]} B4={c[4]} (n={len(rows)})")
    print(f"Bucket-1 rate: {c[1]/len(rows)*100:.1f}%  (need >0 on real 4B run)")
else:
    print("No bucket file — run both conditions first")

In [ ]:
# 8. RISK 1: MBPP+ subprocess scoring
# exec() runs in a worker process — most likely full-run failure point.
MBPP_SMOKE_DIR = "/workspace/runs/smoke_mbpp"
os.makedirs(MBPP_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {MBPP_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks mbppplus \
    --smoke --test \
    --latent_space_realign

mbpp_metas = list(Path(MBPP_SMOKE_DIR).glob("latent_mas/mbppplus/example_*/metadata.json"))
print(f"MBPP+ examples completed: {len(mbpp_metas)}  (want >0, no crash = pass)")
for m in sorted(mbpp_metas):
    d = json.loads(m.read_text())
    print(f"  {m.parent.name}: correct={d.get('correct')}")

In [ ]:
# 9. RISK 2: Resume — run smoke twice, second pass must skip completed examples
OOM_SMOKE_DIR = "/workspace/runs/smoke_resume"
os.makedirs(OOM_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

first_pass = list(Path(OOM_SMOKE_DIR).glob("latent_mas/gsm8k/example_*/metadata.json"))
print(f"First pass: {len(first_pass)} examples written")

import time
t0 = time.time()
!python final_run.py \
    --output_dir {OOM_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign
elapsed = time.time() - t0

print(f"Second pass elapsed: {elapsed:.1f}s  (want <60s — skipping cached examples)")
print(f"Resume working: {elapsed < 60}")

In [ ]:
# 10. RISK 3: Storage budget — project smoke disk usage to full run scale
import shutil

def dir_size_gb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / 1e9

smoke_gb = dir_size_gb(SMOKE_DIR)
scale = (200 / 5) * (17 / 2) * 3  # (full_ex/smoke_ex) * (full_cond/smoke_cond) * tasks
projected_gb = smoke_gb * scale

free_gb = shutil.disk_usage("/workspace").free / 1e9
total_gb = shutil.disk_usage("/workspace").total / 1e9

print(f"Smoke disk:        {smoke_gb:.2f} GB  (5 ex, 2 cond, 1 task)")
print(f"Projected full:    {projected_gb:.1f} GB  (200 ex, 17 cond, 3 tasks)")
print(f"Volume free/total: {free_gb:.0f} / {total_gb:.0f} GB")
print()
if projected_gb > free_gb * 0.8:
    print("⚠️  WARNING: projected usage >80% of free space")
    print("   Add --no_layer_hidden or --no_kv_latent to reduce storage by ~50%")
else:
    print("✅ Storage OK")

In [ ]:
# 11. RISK 4: Untested conditions — topk_gated, random_gated, exp_m_identity_wa
COND_SMOKE_DIR = "/workspace/runs/smoke_conds"
os.makedirs(COND_SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {COND_SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions topk_gated random_gated exp_m_identity_wa \
    --tasks gsm8k \
    --smoke --test \
    --latent_space_realign

print("\n--- Untested condition results ---")
for cond in ["topk_gated", "random_gated", "exp_m_identity_wa"]:
    metas = list(Path(COND_SMOKE_DIR).glob(f"{cond}/gsm8k/example_*/metadata.json"))
    if not metas:
        print(f"  {cond}: NO OUTPUT — crashed or condition name wrong")
        continue
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"  {cond}: {len(metas)} examples, {correct}/{len(metas)} correct")

In [ ]:
# 12. All checks passed — print full run command
FULL_RUN_DIR = "/workspace/runs/full"
print("All smoke tests passed. Run the full pipeline with:")
print()
print(f"python final_run.py \\")
print(f"    --output_dir {FULL_RUN_DIR} \\")
print(f"    --model_name Qwen/Qwen3-4B \\")
print(f"    --tasks gsm8k arc_challenge \\")
print(f"    --latent_space_realign")
print()
print("Then once that completes, add mbppplus:")
print()
print(f"python final_run.py \\")
print(f"    --output_dir {FULL_RUN_DIR} \\")
print(f"    --model_name Qwen/Qwen3-4B \\")
print(f"    --tasks mbppplus \\")
print(f"    --latent_space_realign")